# IWAIT'26 — Weather-aware Heatmap Loss for LongSplat

**이윤호 / KNUVI**

| 섹션 | 내용 |
|------|------|
| 0 | GPU / CUDA 환경 확인 |
| 1 | 레포 클론 및 의존성 설치 |
| 2 | Google Drive 마운트 및 데이터 준비 |
| 3 | Heatmap 생성 (MWFormer / WeatherEdit) |
| 4 | 학습 — Baseline |
| 5 | 학습 — Ours (Heatmap Loss) |
| 6 | 결과 비교 |

> **런타임 설정:** 런타임 → 런타임 유형 변경 → **L4 GPU** 선택

---
## Section 0 — GPU / CUDA 환경 확인

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

---
## Section 1 — 레포 클론 및 의존성 설치

In [ ]:
# 레포 클론 (submodule 포함)
!git clone --recursive https://github.com/Leeyoonho02/IWAIT_26.git
%cd IWAIT_26

In [ ]:
# 기본 패키지 설치
!pip install -q -r requirements.txt

In [ ]:
# CUDA 관련 환경변수 설정 (Colab CUDA 경로)
import os
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# submodule 빌드 및 설치
!pip install -q submodules/simple-knn --no-build-isolation
!pip install -q submodules/diff-gaussian-rasterization --no-build-isolation
!pip install -q submodules/fused-ssim --no-build-isolation

In [ ]:
# (선택) RoPE cuda 커널 컴파일 — 속도 향상, 시간 소요 약 2분
!cd submodules/mast3r/dust3r/croco/models/curope && python setup.py build_ext --inplace
%cd /content/IWAIT_26

---
## Section 2 — Google Drive 마운트 및 데이터 준비

Drive 구조 (예시):
```
MyDrive/IWAIT26/
├── scenes/
│   └── snow_scene/
│       ├── images/      ← 원본 프레임
│       └── heatmaps/    ← Section 3에서 생성 (또는 사전 업로드)
└── mwformer.pth         ← MWFormer 체크포인트
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ★ 여기서 경로 설정
DRIVE_ROOT  = '/content/drive/MyDrive/IWAIT26'
SCENE_NAME  = 'snow_scene'                          # 사용할 씬 이름
SCENE_DIR   = f'{DRIVE_ROOT}/scenes/{SCENE_NAME}'   # images/ 가 여기 안에 있어야 함
MWFORMER_CKPT = f'{DRIVE_ROOT}/mwformer.pth'        # MWFormer 체크포인트

# data/ 심볼릭 링크 생성 (LongSplat 기본 경로)
!mkdir -p data
!ln -sfn {SCENE_DIR} data/{SCENE_NAME}

# 확인
!ls data/{SCENE_NAME}/

---
## Section 3 — Heatmap 생성 (MWFormer / WeatherEdit)

> **사전 생성 옵션:** 연구실 서버에서 미리 생성한 뒤 Drive에 `heatmaps/` 폴더로 올려두면 이 섹션을 건너뛸 수 있다.

WeatherEdit 레포를 클론하고 MWFormer로 각 프레임의 weight map을 `.npy`로 저장.

In [ ]:
# 사전 생성된 heatmaps/ 가 이미 있으면 건너뛰기
import os
heatmap_dir = f'data/{SCENE_NAME}/heatmaps'

if os.path.isdir(heatmap_dir) and len(os.listdir(heatmap_dir)) > 0:
    print(f'[skip] {heatmap_dir} 이미 존재함 ({len(os.listdir(heatmap_dir))}개 파일)')
else:
    print('[info] heatmap 생성 필요 — 아래 셀 실행')

In [ ]:
# WeatherEdit 클론 (MWFormer 소스 포함)
%cd /content
!git clone https://github.com/Jumponthemoon/WeatherEdit.git
%cd /content/IWAIT_26

In [ ]:
# WeatherEdit 의존성 설치 (basicsr 등)
!pip install -q basicsr

In [ ]:
# Heatmap 생성 실행
# --alpha 는 train.py 의 --heatmap_alpha 와 동일한 값으로 맞춰야 함
!python generate_heatmaps.py \
    --scene_dir data/{SCENE_NAME}/images \
    --ckpt      {MWFORMER_CKPT} \
    --out_dir   data/{SCENE_NAME}/heatmaps \
    --alpha     5.0

print('생성된 heatmap 수:', len(os.listdir(f'data/{SCENE_NAME}/heatmaps')))

---
## Section 4 — 학습: Baseline

`heatmaps/` 폴더 없이 실행 → 자동으로 일반 L1 loss (Baseline)

In [ ]:
# Baseline: heatmaps/ 없는 임시 씬 경로 사용
# (또는 data/{SCENE_NAME}/heatmaps 를 잠깐 rename 해도 됨)
BASELINE_OUT = 'output/baseline'

!python train.py \
    -s data/{SCENE_NAME} \
    -m {BASELINE_OUT} \
    --mode custom \
    --resolution 2

# 렌더링 및 지표 측정
!python render.py  -m {BASELINE_OUT}
!python metrics.py -m {BASELINE_OUT}

---
## Section 5 — 학습: Ours (Heatmap Loss)

`data/{SCENE_NAME}/heatmaps/` 폴더가 존재하면 자동 활성화

In [ ]:
# alpha 값 설정 (실험별로 변경)
ALPHA = 5.0
OURS_OUT = f'output/ours_alpha{ALPHA}'

!python train.py \
    -s data/{SCENE_NAME} \
    -m {OURS_OUT} \
    --mode custom \
    --resolution 2 \
    --heatmap_alpha {ALPHA}

# 렌더링 및 지표 측정
!python render.py  -m {OURS_OUT}
!python metrics.py -m {OURS_OUT}

---
## Section 6 — 결과 비교

In [ ]:
import json, glob
import pandas as pd

def load_metrics(output_dir):
    """metrics.py 가 저장한 results.json 을 읽어 dict 반환."""
    paths = glob.glob(f'{output_dir}/**/results.json', recursive=True)
    if not paths:
        return None
    with open(paths[0]) as f:
        data = json.load(f)
    # results.json 구조: {iter: {psnr, ssim, lpips}}
    last = list(data.values())[-1]
    return last

rows = []
for name, out in [('Baseline', BASELINE_OUT), (f'Ours (α={ALPHA})', OURS_OUT)]:
    m = load_metrics(out)
    if m:
        rows.append({'Model': name, **m})
    else:
        rows.append({'Model': name, 'PSNR': '-', 'SSIM': '-', 'LPIPS': '-'})

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())

In [ ]:
# 렌더링 결과 시각화 (첫 번째 프레임 비교)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

def get_render_sample(output_dir):
    imgs = sorted(glob.glob(f'{output_dir}/**/renders/*.png', recursive=True))
    return imgs[0] if imgs else None

baseline_img = get_render_sample(BASELINE_OUT)
ours_img     = get_render_sample(OURS_OUT)

if baseline_img and ours_img:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mpimg.imread(baseline_img)); axes[0].set_title('Baseline'); axes[0].axis('off')
    axes[1].imshow(mpimg.imread(ours_img));     axes[1].set_title(f'Ours (α={ALPHA})'); axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('comparison.png', dpi=150)
    plt.show()
else:
    print('렌더링 결과 없음 — 학습 먼저 실행하세요.')

In [ ]:
# 결과 Drive에 저장
import shutil, os

save_root = f'{DRIVE_ROOT}/results/{SCENE_NAME}'
os.makedirs(save_root, exist_ok=True)

for name, out in [('baseline', BASELINE_OUT), (f'ours_alpha{ALPHA}', OURS_OUT)]:
    dst = f'{save_root}/{name}'
    if os.path.exists(out):
        shutil.copytree(out, dst, dirs_exist_ok=True)
        print(f'[saved] {dst}')

if os.path.exists('comparison.png'):
    shutil.copy('comparison.png', f'{save_root}/comparison.png')
    print(f'[saved] {save_root}/comparison.png')